# Boltz-2 smoke test — EthnoHack Phase 0

One question: **can we predict binding affinity for one known pair, end to end, on Colab's free tier?**

Pass and the mechanism layer stays in the plan. Fail and we drop it now, having spent half an hour instead of a Saturday.

The test pair is carbonic anhydrase II with acetazolamide — a drug and target whose binding is thoroughly measured, so a wild prediction is a visible failure.

**Before running: Runtime → Change runtime type → T4 GPU.** Without a GPU this will not finish.

Neither the protein sequence nor the ligand structure is typed into this notebook; both are fetched from UniProt and PubChem so nothing is transcribed wrongly.

In [ ]:
# 1. Confirm a GPU is attached.
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'NO GPU — Runtime > Change runtime type > T4 GPU'

In [ ]:
# 2. Install Boltz. Takes a few minutes; the weights download on first run.
!pip install -q boltz
!boltz --help | head -20

In [ ]:
# 3. Fetch the inputs rather than typing them.
import requests

UNIPROT = 'P00918'   # human carbonic anhydrase 2
LIGAND = 'acetazolamide'

fasta = requests.get(f'https://rest.uniprot.org/uniprotkb/{UNIPROT}.fasta', timeout=60).text
sequence = ''.join(line.strip() for line in fasta.splitlines()[1:])
print(f'{UNIPROT}: {len(sequence)} residues')
print(sequence[:60], '...')

smiles = requests.get(
    f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{LIGAND}/property/CanonicalSMILES/TXT',
    timeout=60,
).text.strip().splitlines()[0]
print(f'\n{LIGAND}: {smiles}')

In [ ]:
# 4. Write the input file Boltz expects.
from pathlib import Path

yaml = f"""version: 1
sequences:
  - protein:
      id: A
      sequence: {sequence}
  - ligand:
      id: B
      smiles: '{smiles}'
properties:
  - affinity:
      binder: B
"""

Path('smoke').mkdir(exist_ok=True)
Path('smoke/test_pair.yaml').write_text(yaml)
print(yaml[:300], '...')

In [ ]:
# 5. Predict. The MSA step runs on a remote server and is the slow part —
#    expect several minutes, mostly waiting rather than computing.
!boltz predict smoke/test_pair.yaml --use_msa_server --out_dir smoke_out --override

In [ ]:
# 6. Read the answer.
import json, glob

files = glob.glob('smoke_out/**/affinity*.json', recursive=True)
if not files:
    print('No affinity output found. Files produced:')
    for f in glob.glob('smoke_out/**/*', recursive=True)[:40]:
        print(' ', f)
else:
    data = json.load(open(files[0]))
    print(json.dumps(data, indent=2))
    value = data.get('affinity_pred_value')
    prob = data.get('affinity_probability_binary')
    print('\n--- reading ---')
    if prob is not None:
        print(f'probability it binds at all: {prob:.2f}')
    if value is not None:
        print(f'predicted log10(IC50 in uM): {value:.2f}  ->  IC50 ~ {10**value:.3g} uM')
    print('\nAcetazolamide binds carbonic anhydrase II tightly (nanomolar),')
    print('so a sensible prediction is strongly negative on the log10 scale.')
    print('A high binding probability with a low IC50 means PASS.')

## What the result means

**Pass** — a probability near 1 and a small IC50. Boltz-2 works on free Colab, and Saturday's job is batching 20–50 real pairs.

**Out of memory** — the T4's 16GB was not enough for a 260-residue protein. Try a smaller target before giving up; if it still fails, drop the mechanism layer.

**Disconnected mid-run** — the usual Colab timeout. The real run must write each result to Drive as it completes, holding no state in the notebook.

**Nonsense number** — if a known nanomolar binder predicts as weak, the model is not usable for our untested compounds either. That is a finding worth reporting, not a bug to chase.

Record the outcome either way; it decides whether the predicted tier exists.